# Does `CyMirror` behave like a cylinder?

A cylindrical mirror focuses in one plane and does nothing in the other.
This notebook sends a Gaussian beam into a `CyMirror` and checks that the
q parameters gtrace reports for the reflected and transmitted beams are
the ones the theory gives.

The theory is Siegman, *Lasers*, chapter 15, Table 15.1: the ray matrices
of a curved surface at an arbitrary angle of incidence. The entries are
written out again below rather than imported from gtrace, so that what is
being compared is gtrace against the book.

The short version of the answer: it does now. It did not before. A
cylindrical mirror used to be cylindrical in shape only - the two hit
methods kept the cross-section the trace sees and the optical power of
the surface in one variable, and with the curvature out of the plane of
the trace that variable has to be zero, so the power went with it.
Curvature in the plane gave a mirror that focused in *both* planes;
curvature out of it gave one that focused in neither.

In [1]:
import numpy as np

import gtrace.beam as beam
import gtrace.optcomp as opt
import gtrace.optics.gaussian as gauss
from gtrace.optics.geometric import cyl_refl_defl_angle
from gtrace.unit import *

pi = np.pi

WL = 1064*nm
np.set_printoptions(precision=6, suppress=True)

## The theory

For a surface of radius $R$ at an angle of incidence $\theta_1$, going
from index $n_1$ to $n_2$, with $\theta_2$ from Snell's law, Table 15.1
gives, in the reduced-slope convention where every determinant is 1:

*Reflection (d)* - the same surface focuses harder in the plane of
incidence than out of it, because it presents an effective radius
$R\cos\theta$ there and $R/\cos\theta$ perpendicular to it:

$$
M^{\rm refl}_x = \begin{pmatrix} 1 & 0 \\ -\dfrac{2n_1}{R\cos\theta_1} & 1\end{pmatrix}
\qquad
M^{\rm refl}_y = \begin{pmatrix} 1 & 0 \\ -\dfrac{2n_1\cos\theta_1}{R} & 1\end{pmatrix}
$$

*Refraction (f), in the plane of incidence*, and *(g), perpendicular to
it*:

$$
M^{\rm refr}_x = \begin{pmatrix} \dfrac{\cos\theta_2}{\cos\theta_1} & 0 \\[2mm]
\dfrac{n_2\cos\theta_2-n_1\cos\theta_1}{R\cos\theta_1\cos\theta_2} & \dfrac{\cos\theta_1}{\cos\theta_2}\end{pmatrix}
\qquad
M^{\rm refr}_y = \begin{pmatrix} 1 & 0 \\ \dfrac{n_2\cos\theta_2-n_1\cos\theta_1}{R} & 1\end{pmatrix}
$$

A **cylinder** is the surface that presents $R$ to one plane and no
curvature at all to the other, which is $1/R \to 0$ in the entry for that
plane. Siegman says as much on p.616: a cylindrical lens has no focusing
or bending effect on the uncurved coordinate.

Note what that does *not* say. Only for **reflection** does the uncurved
plane become the identity, and only because the angle out equals the
angle in. For **refraction** it does not: a tilted flat interface still
has $A=\cos\theta_2/\cos\theta_1$ and $D=\cos\theta_1/\cos\theta_2$,
because the beam is wider on one side of it than on the other.

In [2]:
def siegman(theta1, n1, n2, invROC_x, invROC_y):
    """Table 15.1, given the curvature each plane sees."""
    c1 = np.cos(theta1)
    theta2 = np.arcsin(n1*np.sin(theta1)/n2)
    c2 = np.cos(theta2)
    Mrx = np.array([[1., 0.], [-2*n1*invROC_x/c1, 1.]])                 # (d)
    Mry = np.array([[1., 0.], [-2*n1*invROC_y*c1, 1.]])                 # (d)
    Mtx = np.array([[c2/c1, 0.],                                        # (f)
                    [(n2*c2 - n1*c1)*invROC_x/(c1*c2), c1/c2]])
    Mty = np.array([[1., 0.], [(n2*c2 - n1*c1)*invROC_y, 1.]])          # (g)
    return Mrx, Mry, Mtx, Mty

## The surface matrices

gtrace's `cyl_refl_defl_angle` takes the curvature of the curved plane
and which plane that is. Compare it with the table, at 45 degrees on a
2 m radius, entering glass.

In [3]:
theta, n1, n2, R = np.deg2rad(45.), 1.0, 1.45, 2.0

def surfaces(curve_direction):
    # The normal points back at the beam, which runs along +x.
    r = cyl_refl_defl_angle(0.0, pi - theta, n1, n2, invROC=1./R,
                            curve_direction=curve_direction)
    return r[2:]

for cd, ix, iy in [('h', 1./R, 0.0), ('v', 0.0, 1./R)]:
    got = surfaces(cd)
    want = siegman(theta, n1, n2, ix, iy)
    print("curve_direction = '%s'" % cd)
    for name, g, w in zip(['Mrx', 'Mry', 'Mtx', 'Mty'], got, want):
        print('  %s  gtrace %s   theory %s   agree %s'
              % (name, np.array2string(g.ravel()),
                 np.array2string(w.ravel()),
                 np.allclose(g, w, rtol=0, atol=1e-15)))
    print()

curve_direction = 'h'
  Mrx  gtrace [ 1.        0.       -1.414214  1.      ]   theory [ 1.        0.       -1.414214  1.      ]   agree True
  Mry  gtrace [ 1.  0. -0.  1.]   theory [ 1.  0. -0.  1.]   agree True
  Mtx  gtrace [1.234656 0.       0.452589 0.809942]   theory [1.234656 0.       0.452589 0.809942]   agree True
  Mty  gtrace [1. 0. 0. 1.]   theory [1. 0. 0. 1.]   agree True

curve_direction = 'v'
  Mrx  gtrace [ 1.  0. -0.  1.]   theory [ 1.  0. -0.  1.]   agree True
  Mry  gtrace [ 1.        0.       -0.707107  1.      ]   theory [ 1.        0.       -0.707107  1.      ]   agree True
  Mtx  gtrace [1.234656 0.       0.       0.809942]   theory [1.234656 0.       0.       0.809942]   agree True
  Mty  gtrace [1.       0.       0.279396 1.      ]   theory [1.       0.       0.279396 1.      ]   agree True



Read the `'v'` row carefully. `Mrx` is the identity - reflection off
the plane with no curvature - but `Mtx` is **not**. It has lost only its
C element. That is the tilt scaling, and it is the half of this that is
easy to get wrong.

## A beam into a mirror

A 1 mm waist at the origin, a metre of free space, and a cylindrical
mirror of 2 m radius at 45 degrees. Three of them: a `Mirror` for
reference, and a `CyMirror` each way round.

In [4]:
Q0 = gauss.Rw2q(np.inf, 1*mm)

def probe():
    return beam.GaussianBeam(q0=Q0, wl=WL, pos=[0.0, 0.0], dirAngle=0.0)

def make(cls, theta, **kw):
    return cls(HRcenter=[1.0, 0], normAngleHR=pi - theta,
               diameter=10*cm, thickness=2*cm, wedgeAngle=0.0,
               inv_ROC_HR=1./R, inv_ROC_AR=0.0,
               Refl_HR=0.5, Trans_HR=0.5, Refl_AR=0.5, Trans_AR=0.5,
               n=1.45, name='M', **kw)

sph = make(opt.Mirror, theta)
cyh = make(opt.CyMirror, theta, curve_direction='h')
cyv = make(opt.CyMirror, theta, curve_direction='v')

# Every one of these has its apex at [1, 0], so the beam reaches all
# three after the same metre and only the matrices differ.
at_mirror = probe()
at_mirror.propagate(1.0)
print('the beam arriving at the mirror')
print('  q  = %s' % at_mirror.qx)
print('  w  = %.6f mm    R = %.6f m'
      % (gauss.q2w(at_mirror.qx, WL)/mm, gauss.q2R(at_mirror.qx)))

the beam arriving at the mirror
  q  = (1+2.952624674426497j)
  w  = 1.055796 mm    R = 9.717992 m


### Reflection

In [5]:
rows = []
for label, m in [('Mirror', sph), ("CyMirror 'h'", cyh), ("CyMirror 'v'", cyv)]:
    r = m.hitFromHR(probe())['r1']
    rows.append((label, r.qx, r.qy))

print('%-14s %-34s %-34s' % ('', 'q in the plane of the trace (x)',
                             'q out of it (y)'))
for label, qx, qy in rows:
    print('%-14s %-34s %-34s' % (label, qx, qy))
print()
print('CyMirror h focuses x as the sphere does :',
      np.isclose(rows[1][1], rows[0][1], rtol=0, atol=1e-15))
print('CyMirror v focuses y as the sphere does :',
      np.isclose(rows[2][2], rows[0][2], rtol=0, atol=1e-15))
print('CyMirror h leaves y as it arrived       :',
      np.isclose(rows[1][2], at_mirror.qy, rtol=0, atol=1e-15))
print('CyMirror v leaves x as it arrived       :',
      np.isclose(rows[2][1], at_mirror.qx, rtol=0, atol=1e-15))

               q in the plane of the trace (x)    q out of it (y)                   
Mirror         (-0.7237412981338769+0.16769075564406474j) (-1.3210226027755174+0.6642899985332854j)
CyMirror 'h'   (-0.7237412981338769+0.16769075564406474j) (0.9999999999999999+2.952624674426497j)
CyMirror 'v'   (1+2.952624674426497j)             (-1.3210226027755174+0.6642899985332854j)

CyMirror h focuses x as the sphere does : True
CyMirror v focuses y as the sphere does : True
CyMirror h leaves y as it arrived       : True
CyMirror v leaves x as it arrived       : True


The two cylinders are not a relabelling of each other: `'h'` in x is
not the same number as `'v'` in y. A sphere focuses harder in the plane
of incidence than out of it, and the cylinder inherits that.

### The focal lengths, and $\cos^2\theta$

Across a thin element $1/q' - 1/q = -1/f$. The in-plane focal length is
$R\cos\theta/2$ and the out-of-plane one $R/(2\cos\theta)$, so they
differ by $\cos^2\theta$ - a factor of two at 45 degrees.

In [6]:
def focal(before, after):
    return -1.0/((1.0/after) - (1.0/before)).real

print('%6s  %12s  %12s  %10s  %10s'
      % ('theta', 'f_h [m]', 'f_v [m]', 'ratio', 'cos^2'))
for deg in [0., 15., 30., 45., 60., 75.]:
    t = np.deg2rad(deg)
    a = probe()
    a.propagate(1.0)
    h = make(opt.CyMirror, t, curve_direction='h').hitFromHR(probe())['r1']
    v = make(opt.CyMirror, t, curve_direction='v').hitFromHR(probe())['r1']
    f_h = focal(a.qx, h.qx)
    f_v = focal(a.qy, v.qy)
    print('%5.1f   %12.9f  %12.9f  %10.7f  %10.7f'
          % (deg, f_h, f_v, f_h/f_v, np.cos(t)**2))
print()
print('theory: f_h = R cos(t)/2, f_v = R/(2 cos(t))')

 theta       f_h [m]       f_v [m]       ratio       cos^2
  0.0    1.000000000   1.000000000   1.0000000   1.0000000
 15.0    0.965925826   1.035276180   0.9330127   0.9330127
 30.0    0.866025404   1.154700538   0.7500000   0.7500000
 45.0    0.707106781   1.414213562   0.5000000   0.5000000
 60.0    0.500000000   2.000000000   0.2500000   0.2500000
 75.0    0.258819045   3.863703305   0.0669873   0.0669873

theory: f_h = R cos(t)/2, f_v = R/(2 cos(t))


### What the mirror does to a beam

The clearest statement of "cylindrical" is what happens downstream. Send
the beam onto a `'h'` mirror at 45 degrees and follow the two widths:
one plane comes to a waist, the other keeps growing as if nothing were
there.

In [7]:
r = cyh.hitFromHR(probe())['r1']

print('%8s  %10s  %10s' % ('z [m]', 'w_x [mm]', 'w_y [mm]'))
print('%8s  %10s  %10s' % ('', '(focused)', '(untouched)'))
for z in np.arange(0.0, 1.61, 0.1):
    b = r.copy()
    b.propagate(z)
    wx = gauss.q2w(b.qx, WL)/mm
    wy = gauss.q2w(b.qy, WL)/mm
    bar = '#'*int(round(wx*12)) + ' '*40
    print('%8.2f  %10.6f  %10.6f  %s' % (z, wx, wy, bar[:34]))

   z [m]    w_x [mm]    w_y [mm]
           (focused)  (untouched)
    0.00    1.055796    1.055796  #############                     
    0.10    0.917909    1.067143  ###########                       
    0.20    0.781538    1.079433  #########                         
    0.30    0.647643    1.092635  ########                          
    0.40    0.518144    1.106717  ######                            
    0.50    0.397365    1.121645  #####                             
    0.60    0.296174    1.137385  ####                              
    0.70    0.240691    1.153906  ###                               
    0.80    0.261800    1.171173  ###                               
    0.90    0.345745    1.189154  ####                              
    1.00    0.459275    1.207817  ######                            
    1.10    0.585424    1.227131  #######                           
    1.20    0.717567    1.247066  #########                         
    1.30    0.852923    1.267593  ##

In [8]:
# The same beam off a spherical mirror, for contrast: both planes
# focus, at slightly different places - astigmatism, not cylindricity.
s = sph.hitFromHR(probe())['r1']
print('%8s  %10s  %10s' % ('z [m]', 'w_x [mm]', 'w_y [mm]'))
for z in np.arange(0.0, 1.61, 0.1):
    b = s.copy()
    b.propagate(z)
    print('%8.2f  %10.6f  %10.6f'
          % (z, gauss.q2w(b.qx, WL)/mm, gauss.q2w(b.qy, WL)/mm))

   z [m]    w_x [mm]    w_y [mm]
    0.00    1.055796    1.055796
    0.10    0.917909    0.992523
    0.20    0.781538    0.930427
    0.30    0.647643    0.869761
    0.40    0.518144    0.810846
    0.50    0.397365    0.754092
    0.60    0.296174    0.700025
    0.70    0.240691    0.649317
    0.80    0.261800    0.602815
    0.90    0.345745    0.561567
    1.00    0.459275    0.526806
    1.10    0.585424    0.499889
    1.20    0.717567    0.482131
    1.30    0.852923    0.474561
    1.40    0.990175    0.477664
    1.50    1.128631    0.491238
    1.60    1.267897    0.514454


## Transmission

Through the substrate, the uncurved plane must lose the power of the
surfaces and keep everything else: the index change, and the tilt
scaling. Across an interface whose matrix is diagonal, gtrace's q scales
by exactly $(n_2/n_1)A^2$, which makes this checkable to the last bit.

In [9]:
N = 1.45
theta2 = np.arcsin(np.sin(theta)/N)

for cd in ['h', 'v']:
    m = make(opt.CyMirror, theta, curve_direction=cd)
    bs = m.hitFromHR(probe(), order=2)
    a = probe()
    a.propagate(bs['input'].length)
    inside = bs['s1'].copy()
    inside.propagate(bs['s1'].length)

    # the plane the cylinder does not curve
    was      = a.qy       if cd == 'h' else a.qx
    in_glass = bs['s1'].qy if cd == 'h' else bs['s1'].qx
    at_ar    = inside.qy  if cd == 'h' else inside.qx
    out      = bs['t1'].qy if cd == 'h' else bs['t1'].qx

    A_in  = 1.0 if cd == 'h' else np.cos(theta2)/np.cos(theta)
    A_out = 1.0 if cd == 'h' else np.cos(theta)/np.cos(theta2)

    print("curve_direction = '%s', the flat plane is %s"
          % (cd, 'y' if cd == 'h' else 'x'))
    print('  entering : ratio %.12f   n*A^2 = %.12f   %s'
          % ((in_glass/was).real, N*A_in**2,
             np.isclose(in_glass, N*A_in**2*was, rtol=0, atol=1e-14)))
    print('  leaving  : ratio %.12f   A^2/n = %.12f   %s'
          % ((out/at_ar).real, A_out**2/N,
             np.isclose(out, A_out**2/N*at_ar, rtol=0, atol=1e-14)))
    print('  end to end, with no 1/R anywhere      :',
          np.isclose(out, A_out**2/N*(N*A_in**2*was + bs['s1'].length),
                     rtol=0, atol=1e-14))
    print()

curve_direction = 'h', the flat plane is y
  entering : ratio 1.450000000000   n*A^2 = 1.450000000000   True
  leaving  : ratio 0.689655172414   A^2/n = 0.689655172414   True
  end to end, with no 1/R anywhere      : True

curve_direction = 'v', the flat plane is x
  entering : ratio 2.210344827586   n*A^2 = 2.210344827586   True
  leaving  : ratio 0.452418096724   A^2/n = 0.452418096724   True
  end to end, with no 1/R anywhere      : True



## Summary

* The surface matrices are Table 15.1, with the curvature given to one
  plane and zero to the other.
* Reflection off the uncurved plane is the identity; refraction through
  it is not, and keeps its A and D.
* The focal lengths in the two planes differ by $\cos^2\theta$, as they
  must.
* A cylindrical mirror brings one plane to a waist and leaves the other
  as it found it.

The same checks run as counted assertions in
`tests/gui/verify_cylindrical.py`, which `tests/gui/run_all.py` drives,
so this notebook is the readable version rather than the authority.